# Feature Engineering

**Objective:** Learn feature engineering with simple, step-by-step examples.

## Feature engineering: give the model useful inputs

A feature is an input variable used by a model. Feature engineering creates, transforms or selects features so that useful structure is easier for an algorithm to learn. A good feature should be available at prediction time and should represent information relevant to the task.

## Feature Engineering
Feature engineering means changing raw columns into inputs that a model can use better.

### Creating and transforming features

Examples include price per square foot, age from date of birth, month from a timestamp, or a ratio between two measurements. Transformations can reduce skew or express a non-linear relationship. Always explain what the new feature means in the real world.

In [1]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

RANDOM_STATE = 42

data = pd.read_csv('datasets/houses.csv')
data

,city,bedrooms,area_sqft,age_years,price
0,Delhi,1,500,5,35
1,Mumbai,2,750,10,60
2,Delhi,3,1200,2,95
3,Pune,2,900,8,75
4,Mumbai,4,1600,1,140
5,Pune,3,1300,4,120


### Categorical encoding

Most classical estimators require numbers. **One-hot encoding** creates a separate indicator column for each category. **Ordinal encoding** maps categories to integer codes when an order is meaningful. Arbitrary integer labels can accidentally suggest an order where none exists.

In [2]:
data['price_per_sqft'] = data['price'] / data['area_sqft']
data['rooms_per_1000_sqft'] = data['bedrooms'] / (data['area_sqft'] / 1000)
data.head()

,city,bedrooms,area_sqft,age_years,price,price_per_sqft,rooms_per_1000_sqft
0,Delhi,1,500,5,35,0.070000,2.000000
1,Mumbai,2,750,10,60,0.080000,2.666667
2,Delhi,3,1200,2,95,0.079167,2.500000
3,Pune,2,900,8,75,0.083333,2.222222
4,Mumbai,4,1600,1,140,0.087500,2.500000


### Scaling numeric values

Standardization subtracts the training mean and divides by the training standard deviation. Min-max scaling maps values into a chosen range, often 0 to 1. Robust scaling uses the median and interquartile range and is less affected by extreme values. Tree-based models are generally less sensitive to monotonic feature scaling than distance- or gradient-based models.

In [3]:
X = data.drop(columns=['price', 'price_per_sqft'])
y = data['price']

numeric_features = ['bedrooms', 'area_sqft', 'age_years', 'rooms_per_1000_sqft']
categorical_features = ['city']

preprocessor = ColumnTransformer([
    ('numeric', StandardScaler(), numeric_features),
    ('categorical', OneHotEncoder(handle_unknown='ignore'), categorical_features)
])

### Feature selection and leakage

Feature selection keeps a subset of useful inputs. It can improve interpretability and reduce noise, but must be performed using training data or within cross-validation. **Leakage** happens when information from validation/test data or the future enters model training or feature construction.

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=RANDOM_STATE)

model = Pipeline([
    ('preprocess', preprocessor),
    ('model', LinearRegression())
])

model.fit(X_train, y_train)
predictions = model.predict(X_test)
print('MAE:', mean_absolute_error(y_test, predictions))

MAE: 20.236335281593696


## Key terms

- **Feature creation:** deriving a new input from existing information.
- **Encoding:** converting categories into numeric representation.
- **Scaling:** changing numeric feature ranges or distributions.
- **Feature selection:** choosing which input columns to keep.
- **Feature extraction:** deriving a new representation, often with dimensionality reduction.
- **Data leakage:** using information that would not legitimately be available at prediction time.

## Try This Yourself
- Create a feature called `is_large_house`.
- Remove scaling and compare the model output.
- Add another categorical value and predict with it.